# 13. Real Data: Joint FWHM + Sigma Optimization

**Goal:** show that our joint optimization (REINFORCE for μ + implicit diff for γ + sigma matching) works on real experimental data, not just synthetic toy data.

**Data:** 3 nW laser power, 40% transmission — the condition with the most valid FWHM measurements (3,913 points, ~18% NaN rate).

In [ ]:
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

from src.fitting import _width, _raw_from_width
from src.samplers import sample_cauchy_truncated
from src.implicit import compute_fwhm_and_dgamma

print('Ready')

In [ ]:
import pandas as pd
df = pd.read_csv('data/processed/fwhm_linewidths.csv')

# Use the best condition: 3 nW, 40% transmission
cond = (df['power_nW'] == 3) & (df['transmission'] == 40)
target_raw = df[cond]['fwhm'].dropna().values

# Scale to match simulator units (simulator works with FWHM ~ 20-60)
SCALE = 1000.0
target_data = torch.tensor(target_raw * SCALE, dtype=torch.float32)

N_TARGET = len(target_data)
print(f'Target: {N_TARGET} FWHM values')
print(f'  Mean: {target_data.mean():.2f}, Std: {target_data.std():.2f}')
print(f'  Range: [{target_data.min():.2f}, {target_data.max():.2f}]')

plt.figure(figsize=(10, 4))
plt.subplot(121)
plt.hist(target_data.numpy(), bins=60, density=True, alpha=0.7)
plt.xlabel('FWHM (scaled MHz)'); plt.ylabel('Density')
plt.title('Real Data: 3 nW, 40% transmission')
plt.grid(alpha=0.3)

plt.subplot(122)
plt.plot(np.sort(target_data.numpy()), np.linspace(0, 1, N_TARGET))
plt.xlabel('FWHM (scaled MHz)'); plt.ylabel('CDF')
plt.title('Empirical CDF')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Optimization Parameters

Same setup as notebook 12, but target comes from real data instead of synthetic generation.

In [ ]:
# Physical parameters
SIGMA_PROP = 6.0        # physical noise std (same as before)
LAMBDA_ = 2.0            # background noise mean (from paper)

# Optimization settings
N_RUNS = 200             # simulations per step
N_ITER = 60              # optimization steps
LR_MU = 3.0
LR_GAMMA = 5.0
LAMBDA_SIGMA = 0.3       # sigma weight in loss
LAMBDA_GAMMA_SIGMA = 2.0 # sigma weight in gamma gradient
CLIP = 10.0
BASELINE_ALPHA = 0.1

# Initial guesses
MU_INIT = 30.0
GAMMA_INIT = 10.0

SEED = 42

# Target quantiles (for quantile matching)
target_sorted = torch.sort(target_data)[0]
target_sig = torch.full((N_TARGET,), target_data.std().item())  # sigma target

# Sub-sample target to match N_RUNS for quantile matching
rng_state = np.random.RandomState(0)
target_idx = rng_state.choice(N_TARGET, N_RUNS, replace=False)
st = target_sorted[torch.sort(torch.tensor(target_idx))[0]]
st_sig = target_sig[torch.sort(torch.tensor(target_idx))[0]]

print(f'Target subsample: {len(st)} points')
print(f'Initial mu={MU_INIT}, gamma={GAMMA_INIT}')

In [ ]:
# ---- Build fit/nll/fwhm wrappers (Lorentzian, no background) ----
from src.fitting import nll as nll_fn, fwhm_from_theta

N_PARAMS = 3  # center, raw_gamma, raw_sigma

def _fit_fn(ph):
    """Fit Lorentzian to photons, return theta*."""
    theta = torch.zeros(N_PARAMS, dtype=torch.float32, requires_grad=True)
    with torch.no_grad():
        theta[0] = 0.0
        theta[1] = _raw_from_width(torch.tensor(10.0))
        theta[2] = _raw_from_width(torch.tensor(5.0))
    opt = torch.optim.LBFGS([theta], max_iter=80, lr=0.5, tolerance_change=1e-8)
    def closure():
        opt.zero_grad()
        loss = nll_fn(theta, ph.detach(), uniform_bg=False)
        loss.backward()
        return loss
    try:
        opt.step(closure)
    except:
        return None
    return theta.detach()

def _fwhm_fn(th):
    return fwhm_from_theta(th)

def _nll_fn(th, ph):
    return nll_fn(th, ph, uniform_bg=False)


# ---- Build photones from reparameterized samples ----
def build_photons(gamma, u, b):
    """Build photon detunings from reparameterized samples."""
    signal = gamma * torch.tan(torch.pi * (torch.tensor(u, dtype=torch.float32) - 0.5))
    # Truncate to window
    mask = (signal >= -75.0) & (signal <= 75.0)
    signal = signal[mask]
    bg = torch.tensor(b, dtype=torch.float32)
    return torch.cat([signal, bg])


# ---- Draw fixed noise ----
def draw_fixed_noise(mu, sigma, lambda_, rng):
    """Draw frozen noise for one run."""
    n = int(round(rng.normal(mu, sigma)))
    n = max(1, n)
    u = rng.uniform(0, 1, size=n).astype(np.float32)
    n_bg = int(rng.poisson(lambda_))
    b = rng.uniform(-75, 75, size=n_bg).astype(np.float32)
    return u, b, n

print('Wrappers ready')

---
## Joint Optimization Loop

Same as notebook 12: μ via REINFORCE, γ via implicit diff + CRLB, with sigma matching.

In [ ]:
mu_val = float(MU_INIT)
gamma_val = float(GAMMA_INIT)
bl = 0.0
history = []

t_start = time.time()
print(f"Optimizing on REAL data (3nW, 40% transmission)...")
print(f"\u03bc0={MU_INIT}, \u03b30={GAMMA_INIT}, target FWHM\u2248{target_data.mean():.1f}")
print(f"N_ITER={N_ITER}, N_RUNS={N_RUNS}\n")

for step in range(N_ITER):
    rng2 = np.random.default_rng(SEED + step)
    fwhms, sigmas, dfs, ns = [], [], [], []

    for _ in range(N_RUNS):
        u, b, n = draw_fixed_noise(mu_val, SIGMA_PROP, LAMBDA_, rng2)
        ns.append(n)
        fw, sig, dg = compute_fwhm_and_dgamma(
            gamma_val, u, b,
            _fit_fn, _fwhm_fn, _nll_fn, n_params=N_PARAMS
        )
        fwhms.append(fw)
        sigmas.append(sig)
        dfs.append(dg)

    ft = torch.tensor(fwhms, dtype=torch.float32)
    si_t = torch.tensor(sigmas, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor(dfs, dtype=torch.float32)

    sf, sidx = torch.sort(ft)
    pl = torch.abs(sf - st[:N_RUNS])
    nss = nt[sidx]
    dgs = dg_t[sidx]
    ss = si_t[sidx]

    pl_sig = torch.abs(ss - st_sig[:N_RUNS])
    loss_fwhm = pl.mean()
    loss_sigma = pl_sig.mean()
    mean_loss = (loss_fwhm + LAMBDA_SIGMA * loss_sigma).item()
    lf_val = loss_fwhm.item()
    ls_val = loss_sigma.item()

    if step == 0:
        bl = mean_loss
    else:
        bl = (1 - BASELINE_ALPHA) * bl + BASELINE_ALPHA * mean_loss

    per_run_reward = -(pl.detach() + LAMBDA_SIGMA * pl_sig.detach())
    adv = (per_run_reward - bl).numpy()
    scores = (nss.numpy() - mu_val) / SIGMA_PROP**2
    raw_grad_mu = float(np.mean(adv * scores))
    grad_mu = max(min(raw_grad_mu, CLIP), -CLIP)
    mu_val += LR_MU * (-grad_mu)
    mu_val = max(1.0, min(200.0, mu_val))

    dsigs = torch.tensor([min(2.0 / math.sqrt(max(int(n), 1)), 2.0) for n in ns])
    ds_s = dsigs[sidx]
    signs_fw = torch.sign(sf - st[:N_RUNS])
    signs_sg = torch.sign(ss - st_sig[:N_RUNS])
    raw_grad_fwhm = float((signs_fw * dgs).mean().item())
    raw_grad_sigma = float((signs_sg * ds_s).mean().item())
    raw_grad_gamma = raw_grad_fwhm + LAMBDA_GAMMA_SIGMA * raw_grad_sigma
    grad_gamma = max(min(raw_grad_gamma, CLIP), -CLIP)
    gamma_val += LR_GAMMA * (-grad_gamma)
    gamma_val = max(0.1, min(100.0, gamma_val))

    rho = float(np.corrcoef(nss.numpy(), pl.numpy())[0, 1]) if pl.std() > 0.01 and nss.std() > 0.01 else 0.0

    info = {
        'step': step, 'mu': mu_val, 'gamma': gamma_val,
        'loss': mean_loss, 'loss_fwhm': lf_val, 'loss_sigma': ls_val,
        'baseline': bl, 'grad_mu': grad_mu, 'grad_gamma': grad_gamma,
        'rho': rho, 'mean_n': float(np.mean(ns)),
        'mean_fwhm': float(ft.mean().item()),
    }
    history.append(info)

    if step % 5 == 0 or step == N_ITER - 1:
        t_elapsed = time.time() - t_start
        print(f"  S{step:2d}: \u03bc={mu_val:6.2f} \u03b3={gamma_val:5.1f} | "
              f"L={mean_loss:.2f}(F={lf_val:.2f}+S={ls_val:.2f}) | "
              f"\u2207\u03bc={grad_mu:+.4f} \u2207\u03b3={grad_gamma:+.4f} | "
              f"n\u0304={info['mean_n']:4.1f} ({t_elapsed:.0f}s)", flush=True)

print(f"\nDone. {time.time()-t_start:.0f}s")

---
## Results: Parameter Development

How μ and γ evolved during optimization.

In [ ]:
if len(history) > 0:
    steps = [h['step'] for h in history]
    mu_vals = [h['mu'] for h in history]
    gamma_vals = [h['gamma'] for h in history]
    losses = [h['loss'] for h in history]
    loss_f = [h['loss_fwhm'] for h in history]
    loss_s = [h['loss_sigma'] for h in history]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    axes[0,0].plot(steps, mu_vals, 'o-', color='steelblue')
    axes[0,0].axhline(float(N_TARGET)/N_RUNS * target_data.mean().item(), 
                     color='gray', ls='--', alpha=0.5, label='target ref')
    axes[0,0].set_xlabel('Step'); axes[0,0].set_ylabel('μ')
    axes[0,0].set_title('μ (mean photon count)'); axes[0,0].grid(alpha=0.3)

    axes[0,1].plot(steps, gamma_vals, 'o-', color='coral')
    axes[0,1].set_xlabel('Step'); axes[0,1].set_ylabel('γ')
    axes[0,1].set_title('γ (Lorentzian HWHM)'); axes[0,1].grid(alpha=0.3)

    axes[0,2].plot(steps, losses, 'o-', color='green', label='Total')
    axes[0,2].plot(steps, loss_f, 'o-', alpha=0.5, label='FWHM')
    axes[0,2].plot(steps, loss_s, 'o-', alpha=0.5, label='Sigma')
    axes[0,2].set_xlabel('Step'); axes[0,2].set_ylabel('Loss')
    axes[0,2].set_title('Loss Components'); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

    axes[1,0].plot(steps, [h['grad_mu'] for h in history], 'o-', color='steelblue')
    axes[1,0].axhline(0, color='gray', ls='--', alpha=0.3)
    axes[1,0].set_xlabel('Step'); axes[1,0].set_ylabel('∇μ')
    axes[1,0].set_title('Gradient μ'); axes[1,0].grid(alpha=0.3)

    axes[1,1].plot(steps, [h['grad_gamma'] for h in history], 'o-', color='coral')
    axes[1,1].axhline(0, color='gray', ls='--', alpha=0.3)
    axes[1,1].set_xlabel('Step'); axes[1,1].set_ylabel('∇γ')
    axes[1,1].set_title('Gradient γ'); axes[1,1].grid(alpha=0.3)

    axes[1,2].plot(steps, [h['mean_fwhm'] for h in history], 'o-', color='purple')
    axes[1,2].axhline(target_data.mean().item(), color='gray', ls='--', 
                      alpha=0.5, label=f'target mean={target_data.mean():.1f}')
    axes[1,2].set_xlabel('Step'); axes[1,2].set_ylabel('Mean FWHM')
    axes[1,2].set_title('Mean FWHM vs Target'); axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Initial vs Final Fit

Compare the initial simulation (step 0) and final simulation (last step) against the real target data.

In [ ]:
if len(history) >= 2:
    # Get first and last simulation outputs
    first_mu = history[0]['mu']
    first_gamma = history[0]['gamma']
    final_mu = history[-1]['mu']
    final_gamma = history[-1]['gamma']

    # Run fresh simulations at initial and final params
    def simulate(mu_val, gamma_val, n_runs=500):
        rng = np.random.default_rng(99)
        fwhms = []
        for _ in range(n_runs):
            u, b, n = draw_fixed_noise(mu_val, SIGMA_PROP, LAMBDA_, rng)
            fw, _, _ = compute_fwhm_and_dgamma(
                gamma_val, u, b,
                _fit_fn, _fwhm_fn, _nll_fn, n_params=N_PARAMS
            )
            fwhms.append(fw)
        return np.array(fwhms)

    print(f"Running 500 simulations at initial params (μ={first_mu:.1f}, γ={first_gamma:.1f})...")
    init_fwhms = simulate(first_mu, first_gamma)
    print(f"Running 500 simulations at final params (μ={final_mu:.1f}, γ={final_gamma:.1f})...")
    final_fwhms = simulate(final_mu, final_gamma)

    target_np = target_data.numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram comparison
    bins = np.linspace(0, max(target_np.max(), init_fwhms.max(), final_fwhms.max()) + 5, 50)
    axes[0].hist(target_np, bins=bins, density=True, alpha=0.5, label=f'Target (real data)', color='black')
    axes[0].hist(init_fwhms, bins=bins, density=True, alpha=0.5, label=f'Initial μ={first_mu:.0f} γ={first_gamma:.0f}', color='red')
    axes[0].hist(final_fwhms, bins=bins, density=True, alpha=0.5, label=f'Final μ={final_mu:.0f} γ={final_gamma:.0f}', color='green')
    axes[0].set_xlabel('FWHM (scaled MHz)'); axes[0].set_ylabel('Density')
    axes[0].set_title('Distribution Comparison')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    # CDF comparison
    axes[1].plot(np.sort(target_np), np.linspace(0, 1, len(target_np)), '-', label='Target (real)', color='black', linewidth=2)
    axes[1].plot(np.sort(init_fwhms), np.linspace(0, 1, len(init_fwhms)), '--', label=f'Initial', color='red')
    axes[1].plot(np.sort(final_fwhms), np.linspace(0, 1, len(final_fwhms)), '--', label=f'Final', color='green', linewidth=2)
    axes[1].set_xlabel('FWHM (scaled MHz)'); axes[1].set_ylabel('CDF')
    axes[1].set_title('CDF Comparison')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\n{'='*60}")
    print(f"  REAL DATA OPTIMIZATION - SUMMARY")
    print(f"{'='*60}")
    print(f"  Target: 3 nW, 40% transmission ({N_TARGET:,} FWHM values)")
    print(f"  Target mean FWHM: {target_data.mean():.2f}")
    print(f"\n  Initial: μ={first_mu:.1f}, γ={first_gamma:.1f}")
    print(f"    Sim FWHM: mean={init_fwhms.mean():.2f}, std={init_fwhms.std():.2f}")
    print(f"  Final:   μ={final_mu:.1f}, γ={final_gamma:.1f}")
    print(f"    Sim FWHM: mean={final_fwhms.mean():.2f}, std={final_fwhms.std():.2f}")
    print(f"\n  Loss: {history[0]['loss']:.2f} → {history[-1]['loss']:.2f}")